# Data Cleaning: Impacts of Data Centers on Electrical Utilities

This notebook loads the raw data from `data/raw/`.

In [23]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

In [24]:
DATA_DIR = Path("data/raw")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## First look at each dataset

### FracTracker U.S. Data Centers

In [25]:
data_centers = pd.read_csv(DATA_DIR / "fractracker_data_centers.csv")
print("Rows and columns:", data_centers.shape)
data_centers.sample(3)

Rows and columns: (1694, 53)


,OBJECTID,GlobalID,facility_id,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,location_determination,purpose,operator_name,tenant,mw,mw_low,mw_high,sizerank,sizerank_numeric,power_source,dedicated_power_plant,number_of_generators,number_of_buildings,cooling_source,cooling_type,facility_size_sqft,property_size_acres,project_cost,expected_date_online,community_pushback,advocacy_information,resistance_status,nda,community_group_website_1,community_group_website_2,petition_url,other_info,editor,information_source,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated,delete_dup
936,3925527,085fa50d-511e-49bd-a8f8-2407a8cf41c7,1780860040054-4476,DC Blox Data Center,648 Grassmere Park,Nashville,TN,37211.0,Davidson,36.090492,-86.74721,Proposed,High,Google Maps and other sources,NaN,NaN,NaN,NaN,NaN,NaN,Unknown,1,Grid (unspecified mix),NaN,NaN,NaN,NaN,NaN,69220.0,NaN,NaN,NaN,Yes,NaN,Organized Advocacy,NaN,Glencliff Neighbors of South Nashville,NaN,https://www.change.org/p/nashville-zoo-says-no...,"160,000 signatures",KE,Media Monitoring,https://www.wkrn.com/news/local-news/nashville...,https://www.wsmv.com/2026/06/04/what-know-abou...,https://www.datacenterdynamics.com/en/news/dc-...,NaN,NaN,NaN,NaN,NaN,1.780790e+12,1.781222e+12,NaN
343,3924934,5b366cfa-0d32-412d-b150-3c91fcd2d3ae,1781567529916-1891,DKL Meta Data Center,2002 E Gurler Rd,DeKalb,IL,60115.0,DeKalb,41.891830,-88.72380,Operating,High,Google Maps and other sources,NaN,Meta,NaN,NaN,NaN,NaN,Hyperscale (100-999 MW),5,NaN,NaN,NaN,5,NaN,NaN,NaN,NaN,$1 billion,NaN,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KE,Crowdsourced,https://datacenters.atmeta.com/2023/11/dekalb-...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.781482e+12,1.781482e+12,NaN
98,3924689,0d14ff42-f24c-4d4a-9351-196e803d30d1,1789747387448-4731,Vantage Aurora campus (Trident / CO21),E-470 and E. 48th Avenue (adjacent to N. Tibet...,Aurora,CO,80206.0,Adams,39.791900,-104.70240,Proposed,Medium,Google Maps only,NaN,Vantage Data Centers,NaN,335-500,334.0,500.0,Hyperscale (100-999 MW),5,Grid (unspecified mix),NaN,NaN,2,Water,Closed loop,860000.0,108.0,NaN,NaN,Yes,NaN,Emerging Concern,NaN,NaN,NaN,https://www.change.org/p/protect-neighbourhood...,NaN,MG,Media Monitoring,https://www.planetizen.com/news/2026/09/138432...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.789690e+12,1.789690e+12,NaN


In [26]:
# Summary Statistics for Data Centers Dataset
# Looking for anything that needs to be cleaned
data_centers.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
OBJECTID,1694.0,NaN,NaN,NaN,3925437.5,489.159994,3924591.0,3925014.25,3925437.5,3925860.75,3926284.0
GlobalID,1694,1694,f23a49a2-5c6c-4b75-bc23-5cdf771d590c,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_id,1694,1694,1779305646902-5353,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
facility_name,1694,1561,H5 Data Center,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,1650,1638,Farm-to-Market 157,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,1690,831,Sterling,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,1694,49,VA,464,NaN,NaN,NaN,NaN,NaN,NaN,NaN
zip,1682.0,NaN,NaN,NaN,43739.790131,26461.220655,1085.0,20166.0,30333.5,72192.0,99734.0
county,1691,477,Loudoun,192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,1694.0,NaN,NaN,NaN,37.604049,4.319988,0.0,33.813575,38.750417,40.06715,70.18478


### FracTracker U.S. Data Centers - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `mw` | Contains both numerical values and ranges. | Separate ranges into `mw_low` and `mw_high` to preserve the available information. |
| `power_source` | String categorical values. | Standardize categories and encode them for modeling. |
| `dedicated_power_plant` | Limited availability across facilities. | Determine if power plants can be identified using location (latitude/longitude) and power source. |
| `number_of_generators` | Contains numerical information stored as strings; limited availability across facilities. | Too limited to be a learnable feature unless more information can be added. |
| `number_of_buildings` | Contains integers, ranges, and strings (e.g., `6+`, `1-19`, `8`, `3 large, 1 small`). | Create a consistent approach for representing ranges and descriptive counts. |
| `cooling_source` | Categorical values stored as strings. | Standardize categories and encode them for modeling. |
| `cooling_type` | Inconsistent categorical formatting (e.g., `Closed loop` vs. `Closed Loop`). | Normalize categorical values and encode them for modeling. |
| `project_cost` | Contains strings and potentially ranges rather than consistent numerical values. | Extract numerical values and determine an appropriate method for representing ranges. |



`mw`

In [27]:
# Remove commas and spaces
mw = data_centers['mw'].str.replace(',', '').str.strip()

# Split and assign ranges
data_centers[['mw_low', 'mw_high']] = mw.str.split('-', expand=True)
data_centers['mw_high'] = data_centers['mw_high'].fillna(data_centers['mw_low']) # fill where no range exists 

# Convert to numeric
data_centers[['mw_low', 'mw_high']] = (
    data_centers[['mw_low', 'mw_high']]
    .apply(pd.to_numeric, errors='coerce')
)

data_centers.loc[data_centers['mw'].notna()][['mw', 'mw_low', 'mw_high']].head()

,mw,mw_low,mw_high
0,3000,3000.0,3000.0
1,120,120.0,120.0
2,1200,1200.0,1200.0
3,300,300.0,300.0
6,500,500.0,500.0


`power_source`

In [28]:
# Normalize power sources
power_source = (
    data_centers['power_source']
    .str.lower()
    .str.strip()
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('/', '_')
    .str.replace(' ', '_')
)

# One-hot encoding for each source type
source_encoded = power_source.str.get_dummies(sep=",_")
data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = [
    "power_source", "batteries", "biomass", "coal", "fuel_cells", 
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid", 
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel", 
    "renewable", "solar", "wind"
]

data_centers.loc[data_centers['power_source'].notna()][columns].sample(5)

,power_source,batteries,biomass,coal,fuel_cells,geothermal,grid_unspecified_mix,hybrid_onsite_and_grid,hydroelectric,natural_gas,nuclear,oil_diesel,renewable,solar,wind
1069,"Natural gas, solar",0,0,0,0,0,0,0,0,1,0,0,0,1,0
13,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
1677,Natural gas,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1079,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0
1660,Grid (unspecified mix),0,0,0,0,0,1,0,0,0,0,0,0,0,0


`number_of_buildings`

In [29]:
# Standardization function for assigning number of buildings
def standardize_buildings(buildings):
    if pd.isna(buildings):
        return np.nan, np.nan
    
    buildings = buildings.replace(' ', '')
    b_low = np.nan
    b_high = np.nan
    
    if '-' in buildings:
        split = buildings.split('-')
        b_low = int(split[0])
        b_high = int(split[1])
        
    elif '+' in buildings:
        buildings = buildings.replace('+', '')
        b_low = int(buildings)
        
    elif buildings.isdigit():
        b_low = int(buildings)
        b_high = int(buildings)
        
    return b_low, b_high
    
# Add new features to handle ranges
data_centers[["buildings_low", "buildings_high"]] = pd.DataFrame(
    data_centers["number_of_buildings"].apply(standardize_buildings).tolist(),
    index=data_centers.index
)

data_centers.loc[data_centers["buildings_low"].notna()][["number_of_buildings", "buildings_low", "buildings_high"]].head()

,number_of_buildings,buildings_low,buildings_high
2,18,18.0,18.0
9,4,4.0,4.0
10,2,2.0,2.0
13,14,14.0,14.0
14,6,6.0,6.0


`cooling_source` + `cooling_type`

In [30]:
for col in ['cooling_source', 'cooling_type']:
    # Normalize cooling cols
    cooling_data = (
        data_centers[col]
        .str.lower()
        .str.strip()
        .str.replace('/', '_')
        .str.replace(' ', '_')
    )

    # One-hot encoding for each source type
    source_encoded = cooling_data.str.get_dummies()
    data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = ['cooling_source', 'air', 'water', 'hybrid_air_water']

data_centers.loc[data_centers['cooling_source'].notna()][columns].sample(5)

,cooling_source,air,water,hybrid_air_water
965,Water,0,1,0
0,Air,1,0,0
242,Water,0,1,0
892,Hybrid air/water,0,0,1
827,Water,0,1,0


In [31]:
columns = ['cooling_type', 'closed_loop', 'fans', 'open_loop']

data_centers.loc[data_centers['cooling_type'].notna()][columns].sample(5)

,cooling_type,closed_loop,fans,open_loop
917,Closed loop,1,0,0
461,Closed loop,1,0,0
766,Closed loop,1,0,0
1016,Closed loop,1,0,0
507,Closed loop,1,0,0


`project_cost`

In [ ]:
def standardize_project_cost(cost):
    if pd.isna(cost):
        return np.nan, np.nan
    
    multiplier = 1
    cost_low = np.nan
    cost_high = np.nan
    
    # Normalize cost
    cost = (
        str(cost)
        .lower()
        .replace(' ', '')
        .replace('$', '')
        .replace(',', '')
        .replace('miillion', 'million')
    )

    # Replace abbreviated m
    cost = re.sub(r'm$', 'million', cost)
    
    if 'million' in cost:
        multiplier = 1000000
    elif 'billion' in cost:
        multiplier = 1000000000
        
    cost = re.sub(r'[mb]illion', '', cost)
    
    # Assign range to low and high
    if '-' in cost:
        split = cost.split('-')
        cost_low = float(split[0]) * multiplier
        cost_high = float(split[1]) * multiplier
        
    elif '+' in cost:
        cost = cost.replace('+', '')
        cost_low = float(cost) * multiplier
        
    else:
        cost_low = float(cost) * multiplier
        cost_high = float(cost) * multiplier
        
    return cost_low, cost_high

# Add new features for handling the cost range
data_centers[["project_cost_low", "project_cost_high"]] = pd.DataFrame(
    data_centers["project_cost"].apply(standardize_project_cost).tolist(),
    index=data_centers.index
)

data_centers[data_centers["project_cost"].notna()][["project_cost", "project_cost_low", "project_cost_high"]].sample(5)

,project_cost,project_cost_low,project_cost_high
938,$400 million,4.000000e+08,4.000000e+08
1009,$800 million,8.000000e+08,8.000000e+08
1154,$1.2 billion,1.200000e+09,1.200000e+09
387,$800 million,8.000000e+08,8.000000e+08
574,$1-1.5 billion,1.000000e+09,1.500000e+09


### FracTracker Crypto Mining Operations

In [33]:
crypto = pd.read_csv(DATA_DIR / "fractracker_crypto.csv")
print("Rows and columns:", crypto.shape)
crypto.sample(3)

Rows and columns: (302, 18)


,Name,City,State,Address,Company,Probable power source,Links,Status,lat,long,Precise Location?,MapCat,GEM_links,Other updates,Date,MW (NYTimes),Unnamed: 16,Unnamed: 17
276,Bitfarms Mercer County facility,North Bend,WA,"49032 Southeast 177th Street, North Bend, WA 9...",NaN,Unknown,https://www.investing.com/news/cryptocurrency-...,Unknown,47.444550,-121.687810,Yes,Current status unknown,NaN,NaN,NaN,NaN,NaN,NaN
70,Coindamental,Rome,GA,"5 Westside Industrial Blvd, Rome, GA 30165",Coindamental,NaN,https://www.romega.us/DocumentCenter/View/2840...,Proposed,34.280324,-85.227035,Yes,Proposed,NaN,NaN,5/17/2024,NaN,NaN,NaN
159,Compute North/Generate Capital Buffalo County ...,Kearney,NE,"3215 Global Dr, Kearney, NE 68847",Generate Capital,"Grid, possible tiny amount of solar",https://www.computenorth.com/news_release/comp...,Operating,40.726111,-99.038718,Yes,Operating / under construction,https://www.gem.wiki/Compute_North_Buffalo_Cou...,NaN,10/16/2023,100 MW,NaN,NaN


In [34]:
crypto.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Name,302,250,CleanSpark,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,283,235,Dalton,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,302,41,GA,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Address,194,189,"430 Davis Rd, Vidalia, GA 30474",4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Company,283,174,CleanSpark,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Probable power source,163,36,Unknown,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Links,299,263,https://www.wbir.com/article/tech/east-tenness...,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,301,11,Operating,138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,302.0,NaN,NaN,NaN,37.830255,5.191345,25.454595,34.456008,36.356141,41.261212,69.71761
long,302.0,NaN,NaN,NaN,-128.525789,651.005258,-11402.57827,-98.202512,-88.774187,-82.477328,-69.170992


### EIA-860 Plant Data

The first row of each EIA-860 Excel file is a title, so the column names are on the second row (`header=1`).

In [35]:
# Added header=1 to skip the first row

plants = pd.read_excel(DATA_DIR / "eia860_2025" / "2___Plant_Y2025.xlsx", header=1)
print("Rows and columns:", plants.shape)
plants.sample(3)

Rows and columns: (17349, 42)


,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,Longitude,NERC Region,Balancing Authority Code,Balancing Authority Name,Name of Water Source,Primary Purpose (NAICS Code),Regulatory Status,Sector,Sector Name,FERC Cogeneration Status,FERC Cogeneration Docket Number,FERC Small Power Producer Status,FERC Small Power Producer Docket Number,FERC Exempt Wholesale Generator Status,FERC Exempt Wholesale Generator Docket Number,Ash Impoundment?,Ash Impoundment Lined?,Ash Impoundment Status,Transmission or Distribution System Owner,Transmission or Distribution System Owner ID,Transmission or Distribution System Owner State,Grid Voltage (kV),Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
9512,58871,Citizens Enterprises Corporation,62052,Citizens Imperial Solar,1102 E Simpson Rd,Calipatria,CA,92233,Imperial,33.191661,-115.433783,WECC,IID,Imperial Irrigation District,NaN,22,NR,2,IPP Non-CHP,N,NaN,Y,QF19-296,N,NaN,NaN,NaN,NaN,Imperial Irrigation District,9216,CA,230,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9263,61402,Hernando Energy,61779,Hernando,14463 Landfill Road,Brooksville,FL,34614,Hernando,28.66898,-82.485733,SERC,SEC,Seminole Electric Cooperative,NaN,22,NR,2,IPP Non-CHP,N,NaN,Y,QF07-390,N,NaN,N,X,NaN,Seminole Electric Cooperative Inc,21554,FL,23,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3739,60637,"Sentinel Peak Resources California, LLC",54449,Dome Project,25605 Westside Hwy,Fellows,CA,93224,Kern,35.220833,-119.583056,WECC,CISO,California Independent System Operator,NaN,211,NR,7,Industrial CHP,Y,88-305-000,Y,88-305-000,N,NaN,N,X,NaN,Pacific Gas & Electric Co,14328,CA,24,,,N,NaN,KERN RIVER GAS TRANSMISSION CO,NaN,NaN,NaN,N,N


In [36]:
plants.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,17349.0,NaN,NaN,NaN,47106.087844,23529.704706,7.0,18642.0,60301.0,64545.0,68045.0
Utility Name,17349,6935,"Altus Power America Management, LLC",247,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,17349.0,NaN,NaN,NaN,52099.079601,22868.185984,1.0,55701.0,61103.0,65921.0,70452.0
Plant Name,17349,17345,Viola,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Street Address,17229,15510,TBD,789,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,17314,6015,Lancaster,78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,17349,51,CA,2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Zip,17349.0,8431.0,93501.0,64.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,17326,1542,Kern,304,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,17349,16666,,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### EIA-860 Generator Data

The generator file has three sheets: `Operable`, `Proposed`, and `Retired and Canceled`. For now we only look at `Operable`.

In [37]:
generators = pd.read_excel(
    DATA_DIR / "eia860_2025" / "3_1_Generator_Y2025.xlsx",
    sheet_name="Operable",
    header=1
)
print("Rows and columns:", generators.shape)
generators.sample(3)

Rows and columns: (27919, 73)


,Utility ID,Utility Name,Plant Code,Plant Name,State,County,Generator ID,Technology,Prime Mover,Unit Code,Ownership,Duct Burners,Can Bypass Heat Recovery Steam Generator?,RTO/ISO LMP Node Designation,RTO/ISO Location Designation for Reporting Wholesale Sales Data to FERC,Nameplate Capacity (MW),Nameplate Power Factor,Summer Capacity (MW),Winter Capacity (MW),Minimum Load (MW),Uprate or Derate Completed During Year,Month Uprate or Derate Completed,Year Uprate or Derate Completed,Status,Synchronized to Transmission Grid,Operating Month,Operating Year,Planned Retirement Month,Planned Retirement Year,Associated with Combined Heat and Power System,Sector Name,Sector,Topping or Bottoming,Energy Source 1,Energy Source 2,Energy Source 3,Energy Source 4,Energy Source 5,Energy Source 6,Startup Source 1,Startup Source 2,Startup Source 3,Startup Source 4,Solid Fuel Gasification System?,Carbon Capture Technology?,Turbines or Hydrokinetic Buoys,Time from Cold Shutdown to Full Load,Fluidized Bed Technology?,Pulverized Coal Technology?,Stoker Technology?,Other Combustion Technology?,Subcritical Technology?,Supercritical Technology?,Ultrasupercritical Technology?,Planned Net Summer Capacity Uprate (MW),Planned Net Winter Capacity Uprate (MW),Planned Uprate Month,Planned Uprate Year,Planned Net Summer Capacity Derate (MW),Planned Net Winter Capacity Derate (MW),Planned Derate Month,Planned Derate Year,Planned New Prime Mover,Planned Energy Source 1,Planned New Nameplate Capacity (MW),Planned Repower Month,Planned Repower Year,Other Planned Modifications?,Other Modifications Month,Other Modifications Year,Multiple Fuels?,Cofire Fuels?,Switch Between Oil and Natural Gas?
18882,5580,"East Kentucky Power Coop, Inc",60863.0,Cooperative Solar One,KY,Clark,PV1,Solar Photovoltaic,PV,NaN,S,X,X,NaN,NaN,8.5,,8.5,8.5,,N,,,OP,X,11.0,2017.0,,,N,Electric Utility,1.0,X,SUN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,N,NaN,NaN
7884,14232,Otter Tail Power Co,7947.0,Solway CT,MN,Beltrami,1,Natural Gas Fired Combustion Turbine,GT,NaN,S,X,X,OTP.SLWAY01,OTP.SLWAY01,50.0,0.85,40.1,47,15,N,,,OP,X,5.0,2003.0,,,N,Electric Utility,1.0,X,NG,DFO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,N,,10M,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,Y,N,Y
21661,63109,"Hales Mills Solar, LLC",63339.0,"Hales Mills Solar, LLC",NY,Fulton,6609,Solar Photovoltaic,PV,NaN,S,X,X,NaN,NaN,5.0,,3.6,2.9,,N,,,OP,X,10.0,2020.0,,,N,IPP Non-CHP,2.0,X,SUN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,,,,,,,,NaN,NaN,,,,NaN,,,NaN,NaN,NaN


In [38]:
generators.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Utility ID,27919.0,5853.0,55938.0,329.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Utility Name,27918,5842,Industrial Power Generating Company LLC,329,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Plant Code,27918.0,NaN,NaN,NaN,42375.107386,26782.287436,1.0,7146.0,56794.5,62643.75,70438.0
Plant Name,27918,14325,Edison Sault,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN
State,27918,51,CA,3161,NaN,NaN,NaN,NaN,NaN,NaN,NaN
County,27918,1479,Los Angeles,424,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Generator ID,27917,10890,1,2923,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Technology,27918,27,Solar Photovoltaic,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Prime Mover,27918,18,PV,7999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Unit Code,2180,310,CC1,770,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### EIA-861M Retail Sales of Electricity


In [39]:
# Added header=2 to skip the first two rows of the Excel sheet

sales = pd.read_excel(DATA_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
print("Rows and columns:", sales.shape)
sales.sample(3)

Rows and columns: (10150, 24)


,Year,Month,State,Data Status,Thousand Dollars,Megawatthours,Count,Cents/kWh,Thousand Dollars.1,Megawatthours.1,Count.1,Cents/kWh.1,Thousand Dollars.2,Megawatthours.2,Count.2,Cents/kWh.2,Thousand Dollars.3,Megawatthours.3,Count.3,Cents/kWh.3,Thousand Dollars.4,Megawatthours.4,Count.4,Cents/kWh.4
7261,2014,9.0,MA,Final,283979.40,1695531.4,2696144.0,16.75,333573.83,2303551.6,398225.0,14.48,81443.73,633899.28,14926.0,12.85,2398.43,26515.52,3.0,9.05,701395.38,4659497.8,3109298.0,15.05
1693,2023,10.0,GA,Final,545312.73,3983155.0,4733001.0,13.69,447149.71,4195708.8,612041.0,10.66,167225.03,2663809.70,24300.0,6.28,698.69,11393.98,1.0,6.13,1160386.20,10854068.0,5369343.0,10.69
4962,2018,6.0,IN,Final,365126.00,2977648.2,2859948.0,12.26,225690.25,2139115.8,354492.0,10.55,279686.06,3786910.80,17756.0,7.39,167.74,1624.10,1.0,10.33,870670.04,8905298.8,3232197.0,9.78


In [40]:
sales.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Year,10150.0,18.0,2025.0,612.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Month,10149.0,NaN,NaN,NaN,6.41206,3.442597,1.0,3.0,6.0,9.0,12.0
State,10149,51,AK,199,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Status,10149,2,Final,9180,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Thousand Dollars,10149.0,NaN,NaN,NaN,322406.257117,374363.877149,13064.0,78291.42,218232.0,404119.32,3266301.5
Megawatthours,10149.0,NaN,NaN,NaN,2358136.574637,2557344.034097,104574.0,595182.0,1706973.5,3135968.7,22481421.0
Count,10149.0,NaN,NaN,NaN,2631403.392847,2731937.969772,226548.0,802281.0,1971280.0,3106777.0,14863374.0
Cents/kWh,10149.0,NaN,NaN,NaN,14.26151,5.25453,6.73,11.08,12.7,15.63,52.72
Thousand Dollars.1,10149.0,NaN,NaN,NaN,251960.525962,316264.755862,20778.0,66788.65,166221.67,313892.57,3327766.4
Megawatthours.1,10149.0,NaN,NaN,NaN,2241268.764867,2416984.780473,126343.74,642345.04,1623061.3,2826659.0,17843545.0


### EIA-930 Balance Data

In [41]:
# This is a subset of the EIA 930 Balance dataset for the second half of 2025

balance = pd.read_csv(DATA_DIR / "eia930" / "EIA930_BALANCE_2025_Jul_Dec.csv")
print("Rows and columns:", balance.shape)
balance.sample(3)

Rows and columns: (267218, 65)


,Balancing Authority,Data Date,Hour Number,Local Time at End of Hour,UTC Time at End of Hour,Demand Forecast (MW),Demand (MW),Net Generation (MW),Total Interchange (MW),Sum(Valid DIBAs) (MW),Demand (MW) (Imputed),Net Generation (MW) (Imputed),Total Interchange (MW) (Imputed),Demand (MW) (Adjusted),Net Generation (MW) (Adjusted),Total Interchange (MW) (Adjusted),Net Generation (MW) from Coal,Net Generation (MW) from Natural Gas,Net Generation (MW) from Nuclear,Net Generation (MW) from All Petroleum Products,Net Generation (MW) from Hydropower Excluding Pumped Storage,Net Generation (MW) from Pumped Storage,Net Generation (MW) from Solar without Integrated Battery Storage,Net Generation (MW) from Solar with Integrated Battery Storage,Net Generation (MW) from Wind without Integrated Battery Storage,Net Generation (MW) from Wind with Integrated Battery Storage,Net Generation (MW) from Battery Storage,Net Generation (MW) from Other Energy Storage,Net Generation (MW) from Unknown Energy Storage,Net Generation (MW) from Geothermal,Net Generation (MW) from Other Fuel Sources,Net Generation (MW) from Unknown Fuel Sources,Net Generation (MW) from Coal (Imputed),Net Generation (MW) from Natural Gas (Imputed),Net Generation (MW) from Nuclear (Imputed),Net Generation (MW) from All Petroleum Products (Imputed),Net Generation (MW) from Hydropower Excluding Pumped Storage (Imputed),Net Generation (MW) from Pumped Storage (Imputed),Net Generation (MW) from Solar without Integrated Battery Storage (Imputed),Net Generation (MW) from Solar with Integrated Battery Storage (Imputed),Net Generation (MW) from Wind without Integrated Battery Storage (Imputed),Net Generation (MW) from Wind with Integrated Battery Storage (Imputed),Net Generation (MW) from Battery Storage (Imputed),Net Generation (MW) from Other Energy Storage (Imputed),Net Generation (MW) from Unknown Energy Storage (Imputed),Net Generation (MW) from Geothermal (Imputed),Net Generation (MW) from Other Fuel Sources (Imputed),Net Generation (MW) from Unknown Fuel Sources (Imputed),Net Generation (MW) from Coal (Adjusted),Net Generation (MW) from Natural Gas (Adjusted),Net Generation (MW) from Nuclear (Adjusted),Net Generation (MW) from All Petroleum Products (Adjusted),Net Generation (MW) from Hydropower Excluding Pumped Storage (Adjusted),Net Generation (MW) from Pumped Storage (Adjusted),Net Generation (MW) from Solar without Integrated Battery Storage (Adjusted),Net Generation (MW) from Solar witho Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind without Integrated Battery Storage (Adjusted),Net Generation (MW) from Wind with Integrated Battery Storage (Adjusted),Net Generation (MW) from Battery Storage (Adjusted),Net Generation (MW) from Other Energy Storage (Adjusted),Net Generation (MW) from Unknown Energy Storage (Adjusted),Net Generation (MW) from Geothermal (Adjusted),Net Generation (MW) from Other Fuel Sources (Adjusted),Net Generation (MW) from Unknown Fuel Sources (Adjusted),Region
209199,SPA,09/05/2025,23,09/05/2025 11:00:00 PM,09/06/2025 4:00:00 AM,17.0,15.0,580.0,565.0,565.0,NaN,NaN,NaN,15.0,580.0,565.0,NaN,NaN,NaN,NaN,580.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,580.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CENT
67173,FMPP,08/08/2025,10,08/08/2025 10:00:00 AM,08/08/2025 2:00:00 PM,2718.0,2537.0,2333.0,-204.0,-204.0,NaN,NaN,NaN,2537.0,2333.0,-204.0,310.0,1680.0,129.0,0.0,NaN,NaN,198.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,310.0,1680.0,129.0,0.0,NaN,NaN,198.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,NaN,FLA
232647,TEPC,11/01/2025,19,11/01/2025 7:00:00 PM,11/02/2025 2:00:00 AM,1740.0,1745.0,1737.0,-8.0,-8.0,NaN,NaN,NaN,1745.0,1737.0,-8.0,337.0,1166.0,NaN,NaN,NaN,NaN,0.0,NaN,17.0,NaN,217.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,337.0,1166.0,NaN,NaN,NaN,NaN,0.0,NaN,17.0,Na

In [42]:
balance.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Balancing Authority,267218,61,AECI,4417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Date,267218,184,11/02/2025,1490,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour Number,267218.0,NaN,NaN,NaN,12.502339,6.923663,1.0,7.0,13.0,19.0,25.0
Local Time at End of Hour,267218,4416,11/02/2025 1:00:00 AM,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UTC Time at End of Hour,267218,4420,07/01/2025 8:00:00 AM,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Demand Forecast (MW),233431.0,NaN,NaN,NaN,9465.17759,19303.64298,-227966.0,791.0,2449.0,6493.0,156738.0
Demand (MW),233639.0,NaN,NaN,NaN,9503.464246,19220.239208,-39388.0,815.0,2544.0,6665.0,810948.0
Net Generation (MW),266717.0,NaN,NaN,NaN,8304.861227,18525.972929,-2576.0,547.0,1759.0,5935.0,810705.0
Total Interchange (MW),266739.0,NaN,NaN,NaN,-78.283611,1344.438016,-11031.0,-496.0,-8.0,387.0,40651.0
Sum(Valid DIBAs) (MW),265658.0,NaN,NaN,NaN,-60.813783,1416.714082,-57015.0,-505.0,-13.0,374.0,40651.0


### EIA-930 Subregion Data

In [43]:
subregion = pd.read_csv(DATA_DIR / "eia930" / "EIA930_SUBREGION_2025_Jul_Dec.csv")
print("Rows and columns:", subregion.shape)
subregion.sample(3)

Rows and columns: (366596, 7)


,Balancing Authority,Data Date,Hour Number,Sub-Region,Demand (MW),Local Time at End of Hour,UTC Time at End of Hour
172661,PJM,07/20/2025,6,PS,4983.0,07/20/2025 6:00:00 AM,07/20/2025 10:00:00 AM
190103,PJM,08/25/2025,15,AE,1534.0,08/25/2025 3:00:00 PM,08/25/2025 7:00:00 PM
161433,NYIS,12/24/2025,12,ZONB,966.0,12/24/2025 12:00:00 PM,12/24/2025 5:00:00 PM


In [44]:
subregion.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Balancing Authority,366596,8,PJM,88340,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Data Date,366596,184,11/02/2025,2060,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hour Number,366596.0,NaN,NaN,NaN,12.502319,6.923647,1.0,7.0,13.0,19.0,25.0
Sub-Region,366596,83,PGAE,4417,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Demand (MW),365441.0,NaN,NaN,NaN,3942.233748,4968.20665,-4.0,814.0,1889.0,4780.0,32436.0
Local Time at End of Hour,366596,4416,11/02/2025 1:00:00 AM,151,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UTC Time at End of Hour,366596,4420,07/01/2025 8:00:00 AM,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN
